# PillSeek — imprint reader: SMALL vs BASE (speed models)
Runtime → **A100 GPU**. Run cells top to bottom (~1.5 h). Upload `manifest.json` in cell 2; phone photos in cell 5.


In [ ]:
# 1) Install
!pip -q install "transformers==4.46.3" "tokenizers<0.21" sentencepiece protobuf accelerate jiwer
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
# 2) Upload manifest.json
from google.colab import files
up = files.upload()
assert 'manifest.json' in up

In [ ]:
# 3) Download images (20-40 min)
import json, os, re, requests, hashlib
from concurrent.futures import ThreadPoolExecutor
rows = json.load(open('manifest.json', encoding='utf-8-sig'))
os.makedirs('imgs', exist_ok=True)
def norm_imprint(s):
    toks = [t for t in re.split(r'[;,\s]+', s.upper()) if t]
    return ' '.join(toks)
for r in rows:
    r['path'] = os.path.join('imgs', hashlib.md5(r['url'].encode()).hexdigest() + '.jpg')
    r['text'] = norm_imprint(r['imprint'])
rows = [r for r in rows if 1 <= len(r['text']) <= 40]
print(f'{len(rows)} labeled images')
def fetch(r):
    if os.path.exists(r['path']) and os.path.getsize(r['path']) > 0: return 0
    try:
        open(r['path'], 'wb').write(requests.get(r['url'], timeout=30).content); return 1
    except Exception: return -1
with ThreadPoolExecutor(16) as ex: res = list(ex.map(fetch, rows))
rows = [r for r in rows if os.path.exists(r['path']) and os.path.getsize(r['path']) > 1000]
print(f'downloaded {res.count(1)}, cached {res.count(0)}, failed {res.count(-1)}; usable {len(rows)}')

In [ ]:
# 3b) Drop unreadable image files
from PIL import Image
good = []
for r in rows:
    try:
        Image.open(r['path']).convert('RGB'); good.append(r)
    except Exception: pass
print(f'kept {len(good)} of {len(rows)}'); rows = good


In [ ]:
# 4) Fine-tune BOTH small and base, evaluate each (about 60-90 min total on A100)
import random, io, jiwer
from PIL import Image, ImageFilter, ImageOps, ImageEnhance
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.utils.data import Dataset, DataLoader

random.seed(0); random.shuffle(rows)
slugs = sorted({r['slug'] for r in rows}); random.shuffle(slugs)
held = set(slugs[: max(200, len(slugs)//20)])
train_rows = [r for r in rows if r['slug'] not in held]
val_rows = [r for r in rows if r['slug'] in held]
print(f'train {len(train_rows)}  val {len(val_rows)} (held-out pills: {len(held)})')

def phone_aug(img):
    if random.random() < 0.5:
        w = random.randint(320, 700); img = img.resize((w, int(w*img.height/img.width)), Image.BILINEAR)
    if random.random() < 0.4: img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 1.5)))
    if random.random() < 0.5:
        buf = io.BytesIO(); img.save(buf, 'JPEG', quality=random.randint(35, 85)); img = Image.open(io.BytesIO(buf.getvalue())).convert('RGB')
    if random.random() < 0.5: img = img.rotate(random.uniform(-25, 25), fillcolor=(128,128,128), expand=False)
    if random.random() < 0.5: img = ImageOps.autocontrast(img, cutoff=random.randint(0, 5))
    if random.random() < 0.3: img = ImageEnhance.Brightness(img).enhance(random.uniform(0.6, 1.4))
    return img

RESULTS = {}
for tag, MODEL, BS, EPOCHS, LR in (('small', 'microsoft/trocr-small-printed', 64, 6, 5e-5),
                                  ('base',  'microsoft/trocr-base-printed',  48, 5, 3e-5)):
    print(f'\n================ {tag}: {MODEL} ================')
    processor = TrOCRProcessor.from_pretrained(MODEL, use_fast=False)
    model = VisionEncoderDecoderModel.from_pretrained(MODEL).cuda()
    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.eos_token_id = processor.tokenizer.sep_token_id
    model.config.use_cache = True
    model.generation_config.max_length = 24; model.generation_config.num_beams = 2

    class DS(Dataset):
        def __init__(self, rs, train): self.rs, self.train = rs, train
        def __len__(self): return len(self.rs)
        def __getitem__(self, i):
            r = self.rs[i]
            img = Image.open(r['path']).convert('RGB')
            if self.train: img = phone_aug(img)
            pv = processor(images=img, return_tensors='pt').pixel_values[0]
            labels = processor.tokenizer(r['text'], padding='max_length', max_length=24, truncation=True).input_ids
            labels = [l if l != processor.tokenizer.pad_token_id else -100 for l in labels]
            return pv, torch.tensor(labels)

    dl = DataLoader(DS(train_rows, True), batch_size=BS, shuffle=True, num_workers=4, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS*len(dl), pct_start=0.1)
    model.train()
    for ep in range(EPOCHS):
        tot = 0.0
        for step, (pv, labels) in enumerate(dl):
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                loss = model(pixel_values=pv.cuda(), labels=labels.cuda()).loss
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
            tot += loss.item()
        print(f'  {tag} epoch {ep+1}/{EPOCHS} avg loss {tot/len(dl):.3f}')
    model.eval()
    out_dir = f'pill_trocr_{tag}'
    model.save_pretrained(out_dir); processor.save_pretrained(out_dir)

    def read(img):
        pv = processor(images=img, return_tensors='pt').pixel_values.cuda()
        with torch.no_grad(): ids = model.generate(pv, max_length=24, num_beams=2, min_new_tokens=1)
        return processor.batch_decode(ids, skip_special_tokens=True)[0].strip().upper()
    exact = tok_hits = tok_total = 0
    sample = val_rows[:600]
    for r in sample:
        pred = read(Image.open(r['path']).convert('RGB'))
        if pred == r['text']: exact += 1
        gt = set(r['text'].split()); pr = set(pred.split())
        tok_hits += len(gt & pr); tok_total += len(gt)
    RESULTS[tag] = (100*exact/len(sample), 100*tok_hits/tok_total)
    print(f'>>> {tag}: exact imprint {RESULTS[tag][0]:.1f}%   token recall {RESULTS[tag][1]:.1f}%   (n={len(sample)})')
    globals()[f'reader_{tag}'] = read

print('\nSUMMARY (large model from round 1 was: exact 73.2%, token recall 81.2%)')
for tag, (ex, tr) in RESULTS.items(): print(f'  {tag:<6} exact {ex:.1f}%   token recall {tr:.1f}%')


In [ ]:
# 5) Read your phone photos with BOTH models (upload the same pill photos as before)
from google.colab import files
ups = files.upload()
for name in ups:
    img = Image.open(name).convert('RGB')
    print(f"{name}:  small -> {reader_small(img)!r}   base -> {reader_base(img)!r}")


In [ ]:
# 6) Package + download both (small ~250MB, base ~1.3GB). Use the Files sidebar if downloads are blocked.
!zip -qr pill_trocr_small.zip pill_trocr_small
!zip -qr pill_trocr_base.zip pill_trocr_base
from google.colab import files
files.download('pill_trocr_small.zip'); files.download('pill_trocr_base.zip')
